# Crime Risk Analysis Mexico
## 01 — Data Exploration

This notebook performs an initial exploration and validation of the
SESNSP municipal crime dataset used in the project.

### Objectives

- Inspect the structure and dimensions of the dataset.
- Review data types and missing values.
- Identify duplicated records.
- Validate the temporal and geographic coverage.
- Detect anomalous values before loading and transforming the data in SQL Server.

In [5]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_PATH = Path("../data/raw/Municipal-Delitos-2015-2025_ago2026.csv")

In [10]:
DATA_PATH = Path(
    "../data/raw/Municipal-Delitos-2015-2025_ago2026.csv"
)

print("File exists:", DATA_PATH.exists())

File exists: True


In [11]:
df = pd.read_csv(
    DATA_PATH,
    encoding="latin1",
    low_memory=False
)

df.head()

,Año,Clave_Ent,Entidad,Cve. Municipio,Municipio,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,2,0,1,1,0,1,1,0,2,1,0,1
1,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0,1,0,0,0,0
2,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,0,1,1,3,2,0,1,2,0,0,0
3,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,1,0,0,1,0,0,0,0,0,0,0,0
4,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,0,0,0,1,0,0,0,0,0,0,0


In [12]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 2,562,994
Columns: 21


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2562994 entries, 0 to 2562993
Data columns (total 21 columns):
 #   Column                  Dtype
---  ------                  -----
 0   Año                     int64
 1   Clave_Ent               int64
 2   Entidad                 str  
 3   Cve. Municipio          int64
 4   Municipio               str  
 5   Bien jurídico afectado  str  
 6   Tipo de delito          str  
 7   Subtipo de delito       str  
 8   Modalidad               str  
 9   Enero                   int64
 10  Febrero                 int64
 11  Marzo                   int64
 12  Abril                   int64
 13  Mayo                    int64
 14  Junio                   int64
 15  Julio                   int64
 16  Agosto                  int64
 17  Septiembre              int64
 18  Octubre                 int64
 19  Noviembre               int64
 20  Diciembre               int64
dtypes: int64(15), str(6)
memory usage: 665.7 MB


In [14]:
null_values = df.isnull().sum()

null_values[null_values > 0]

Series([], dtype: int64)

In [15]:
duplicate_rows = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows:,}")

Duplicate rows: 0


In [16]:
print("Years:")
print(sorted(df["Año"].unique()))

print("\nNumber of entities:")
print(df["Entidad"].nunique())

Years:
[np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Number of entities:
32


In [17]:
month_columns = [
    "Enero", "Febrero", "Marzo", "Abril",
    "Mayo", "Junio", "Julio", "Agosto",
    "Septiembre", "Octubre", "Noviembre", "Diciembre"
]

In [18]:
negative_values = (df[month_columns] < 0).sum()

negative_values[negative_values > 0]

Mayo          1
Septiembre    1
dtype: int64

In [19]:
negative_rows = df[
    (df[month_columns] < 0).any(axis=1)
]

negative_rows

,Año,Clave_Ent,Entidad,Cve. Municipio,Municipio,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
404869,2017,9,Ciudad de México,9006,Iztacalco,Libertad personal,Otros delitos que atentan contra la libertad p...,Otros delitos que atentan contra la libertad p...,Otros delitos que atentan contra la libertad p...,0,3,1,3,6,3,5,0,-1,3,1,1
814520,2018,27,Tabasco,27011,Jonuta,El patrimonio,Robo,Robo de vehículo automotor,Robo de coche de 4 ruedas Sin violencia,0,0,0,0,-1,0,0,0,3,1,1,0


### Data quality finding: negative crime counts

Two negative values were identified across the monthly crime-count columns.

Because negative crime counts are not analytically valid and the available
documentation does not provide enough information to interpret these values,
the original records are preserved in the raw dataset.

During the SQL transformation stage, negative values are converted to `NULL`
in the analytical view rather than being replaced with zero.

This approach preserves the source data while preventing anomalous values
from affecting subsequent aggregations and analysis.